# Data Cleaning

**Objective:** Handle data quality issues identified in `data-understanding.ipynb` and produce a clean dataset for analysis.

**Issues found during data understanding:**
- 3 columns have missing values: `smoking_status` (3.1%), `alcohol_consumption` (25.3%), `income_bracket` (3.1%)
- No duplicates or data type issues were found

**What this notebook covers:**
1. Fill missing categorical values with "Unknown"
2. Verify no missing values remain
3. Validate numeric ranges for clinical plausibility
4. Validate categorical levels for consistency
5. Export the cleaned dataset

**Output:** `data/diabetes_risk_cleaned.csv`

**Next step ->** Proceed to `eda.ipynb` for exploratory data analysis.

---
## Load Raw Data

In [24]:
import pandas as pd

df = pd.read_csv("data/diabetes_risk.csv")

---
## 1. Fill Missing Values with "Unknown"

**Columns affected:** `smoking_status`, `alcohol_consumption`, `income_bracket`

**Why "Unknown" instead of other approaches:**

| Approach | Why not? |
|---|---|
| **Drop rows** | Would lose up to 25% of the dataset (3,788 rows for `alcohol_consumption` alone) |
| **Fill with mode** (e.g. "Never") | Would fabricate data — we don't know these patients' actual status |
| **Fill with "Unknown"** | Honest, preserves all 15,000 rows, and lets the model learn that missingness itself may be informative |

In [25]:
cols_to_fill = ['smoking_status', 'alcohol_consumption', 'income_bracket']
df[cols_to_fill] = df[cols_to_fill].fillna('Unknown')

---
## 2. Verify No Missing Values Remain

Confirm the fill worked — all three columns should now show 0 missing.

In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   patient_id                15000 non-null  int64  
 1   age                       15000 non-null  int64  
 2   gender                    15000 non-null  str    
 3   city                      15000 non-null  str    
 4   bmi                       15000 non-null  float64
 5   family_history_diabetes   15000 non-null  str    
 6   physical_activity_level   15000 non-null  str    
 7   diet_type                 15000 non-null  str    
 8   smoking_status            15000 non-null  str    
 9   alcohol_consumption       15000 non-null  str    
 10  hours_sleep_per_night     15000 non-null  float64
 11  stress_level              15000 non-null  int64  
 12  fasting_blood_sugar       15000 non-null  int64  
 13  hba1c_level               15000 non-null  float64
 14  blood_pressure_sy

**Result:** All 19 columns now show 15,000 non-null values. No missing data remains.

In [27]:
df[cols_to_fill].isnull().sum()

smoking_status         0
alcohol_consumption    0
income_bracket         0
dtype: int64

---
## 3. Validate Numeric Ranges

Check that all numeric columns fall within clinically plausible ranges. Flag anything that looks like a data entry error.

| Column | Expected Range |
|---|---|
| `age` | 18 - 120 |
| `bmi` | 10 - 60 |
| `fasting_blood_sugar` | 50 - 400 mg/dL |
| `hba1c_level` | 3.0 - 15.0% |
| `blood_pressure_systolic` | 70 - 250 |
| `blood_pressure_diastolic` | 40 - 140 |
| `hours_sleep_per_night` | 0 - 24 |
| `waist_circumference_cm` | 50 - 180 |

In [28]:
df.describe()

,patient_id,age,bmi,hours_sleep_per_night,stress_level,fasting_blood_sugar,hba1c_level,blood_pressure_systolic,blood_pressure_diastolic,waist_circumference_cm
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,7500.500000,44.333667,23.062260,7.309860,5.941200,168.789933,6.878387,139.496200,86.256933,100.207933
min,1.000000,18.000000,15.000000,3.000000,1.000000,72.000000,4.800000,80.000000,50.000000,73.200000
max,15000.000000,80.000000,40.800000,12.000000,10.000000,400.000000,13.400000,200.000000,120.000000,148.200000


**Result:** All min/max values fall within their expected clinical ranges. No outliers or data entry errors detected.

---
## 4. Validate Categorical Levels

Print the unique values for each categorical column. Look for:
- **Typos** (e.g. "Feamle" vs "Female")
- **Inconsistent casing** (e.g. "high" vs "High")
- **Unexpected categories** that should not exist

In [30]:
for col in df.select_dtypes(include='object').columns:
    print(f"{col}: {df[col].unique()}")

gender: ['Female', 'Male', 'Other']
city: ['Mumbai', 'Thane', 'Bengaluru', 'Hyderabad', 'Delhi', 'Kolkata', 'Ahmedabad', 'Pune', 'Chennai', 'Kanpur', 'Bhopal', 'Visakhapatnam', 'Patna', 'Jaipur', 'Surat', 'Nagpur', 'Indore', 'Lucknow']
family_history_diabetes: ['Yes', 'No']
physical_activity_level: ['Sedentary', 'Moderate', 'Active']
diet_type: ['Vegetarian', 'Non-Vegetarian', 'Pescatarian', 'Vegan']
smoking_status: ['Never', 'Current', 'Unknown', 'Former']
alcohol_consumption: ['Never', 'Regular', 'Occasional', 'Unknown']
income_bracket: ['High', 'Low', 'Middle', 'Unknown']
diabetes_risk: ['Low', 'High', 'Moderate']


**Result:** All categorical levels are clean and consistent — no typos, no casing issues, no unexpected values. The "Unknown" category now appears in the three columns where we filled missing values.

---
## 5. Export Cleaned Dataset

Save the cleaned dataframe to CSV. This file is the input for all downstream notebooks (`eda.ipynb`, modelling, etc.).

**Important:** We use `index=False` to avoid writing the pandas row index as an extra column.

In [31]:
df.to_csv('data/diabetes_risk_cleaned.csv', index=False)
print(f"Cleaned data saved: {df.shape}")

Cleaned data saved: (15000, 19)


---
## Summary of Changes

| Step | Action | Result |
|---|---|---|
| Missing values | Filled `smoking_status`, `alcohol_consumption`, `income_bracket` with "Unknown" | 0 missing values remaining |
| Numeric ranges | Validated against clinical bounds | All within range |
| Categorical levels | Checked for typos and casing | All clean |
| Export | Saved to `data/diabetes_risk_cleaned.csv` | 15,000 rows x 19 columns |

**Next step ->** Proceed to `eda.ipynb` for exploratory data analysis.